# Explore: 낯선 코드베이스에서 근거 확보하기

이 노트북은 에이전트를 처음 보는 저장소에 떨어뜨려 놓고 실제 아키텍처를 파악하게 합니다. 파일 시스템이 에이전트의 유일한 작업 공간이고, 에이전트가 읽기로 선택한 파일만 컨텍스트 윈도에 들어옵니다. 그래서 `ls`, `grep`, `read`로 탐색하며 머릿속 모형을 만들어 가게 됩니다.

흥미로운 부분은 픽스처에 심어 둔 함정입니다. `ARCHITECTURE.md`가 코드에서 더는 따르지 않는 구조를 설명하고 있어서, 코드를 확인하지 않고 문서를 믿는 에이전트는 자신 있게 틀린 답을 내놓게 됩니다. 여기서 말하는 "근거 확보(grounding)"란 문서를 권위 있는 것으로 받아들이는 대신, 읽은 내용을 실제로 존재하는 것과 대조해 검증하는 일입니다.

iterate 노트북에 더해 이 노트북이 가르쳐 주는 것:

- **행동에 앞선 탐색.** 좋은 에이전트는 트리를 충분히 읽어 이해한 뒤에 답합니다. 그 반대가 아닙니다.
- **세션 도중 리소스 추가하기.** 마지막의 사이드바에서는 세션을 다시 만들지 않고 `sessions.resources.add`로 실행 중인 세션에 파일을 더 넣는 방법을 보여 줍니다. 탐색 중에 에이전트가 다음으로 봐야 할 것이 드러났을 때 유용합니다.

In [ ]:
import io
import os

from anthropic import Anthropic
from utilities import (
    make_unfamiliar_repo_zip,
    stream_until_end_turn,
    wait_for_idle_status,
)

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")

client = Anthropic()

## 1. 저장소 픽스처 생성하기

저장소가 작아서 노트북 옆에 디스크 픽스처를 두는 대신 헬퍼로 메모리에서 만듭니다. 이 헬퍼는 `services/` 마이크로서비스 구조를 만들고, 예전 모놀리식 구조를 여전히 설명하고 있는 낡은 `ARCHITECTURE.md`를 심어 둡니다.

In [ ]:
buf = make_unfamiliar_repo_zip()
fixture_zip = client.beta.files.upload(file=("repo.zip", buf, "application/zip"))
print(f"fixture: {fixture_zip.id}")

## 2. 에이전트 + 환경 + 세션

In [ ]:
agent = client.beta.agents.create(
    name="cookbook-explore",
    model=MODEL,
    system=(
        "You are onboarding to an unfamiliar codebase. Explore before "
        "answering, docs can be stale. Verify what you read against "
        "actual code structure. Write notes to /tmp/NOTES.md as you go."
    ),
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
        }
    ],
)

env = client.beta.environments.create(
    name="cookbook-explore-env",
    config={"type": "cloud", "networking": {"type": "limited"}},
)

session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    resources=[{"type": "file", "file_id": fixture_zip.id, "mount_path": "repo.zip"}],
    title="Onboard to repo",
)
print(f"session: {session.id}")

## 3. 탐색하며 낡은 문서 함정 살펴보기

근거를 갖춘 답이라면 실제 `services/` 구조를 언급하고 `ARCHITECTURE.md`가 낡았다고 지적합니다. 근거 없는 답은 낡은 문서가 설명하는 모놀리식 구조를 그대로 읊습니다.

In [ ]:
client.beta.sessions.events.send(
    session_id=session.id,
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Unzip /mnt/session/uploads/repo.zip to /tmp/repo/. "
                        "Then: what is the actual architecture of this "
                        "codebase? Be specific about directory structure. "
                        "Check if the docs are accurate."
                    ),
                }
            ],
        }
    ],
)

print("=== exploration ===")
stream_until_end_turn(client, session.id)

## 4. 에이전트의 메모 읽어 보기

에이전트에게 작업하면서 `/tmp/NOTES.md`에 메모를 남기라고 했습니다. 이 파일을 출력해 보면 탐색 과정에서 코드베이스에 대한 이해가 어떻게 발전했는지 볼 수 있습니다.

In [ ]:
client.beta.sessions.events.send(
    session_id=session.id,
    events=[
        {
            "type": "user.message",
            "content": [{"type": "text", "text": "cat /tmp/NOTES.md"}],
        }
    ],
)
stream_until_end_turn(client, session.id)

## 사이드바: 실행 중인 세션에 컨텍스트 더 넣기

`sessions.create`의 `resources=` 인자가 파일을 마운트하는 가장 흔한 방법이지만, API는 기존 세션의 마운트를 관리하는 `/v1/sessions/<id>/resources` 하위 리소스도 제공합니다. 여기서 유용합니다. 탐색 중에 추가 맥락(설정 파일, 변경 이력, 외부 스키마)이 필요한 질문이 생기면, 세션을 허물지 않고 그것을 넣어 줄 수 있습니다.

패턴은 이미 알고 있는 업로드 후 연결하기 루프와 같고, 다만 한 번의 호출이 두 번으로 나뉠 뿐입니다:

In [ ]:
hints = b"# DEPLOY HISTORY\n2026-03-01: monolith -> microservices migration complete\n"
hints_file = client.beta.files.upload(
    file=("DEPLOY_HISTORY.md", io.BytesIO(hints), "text/markdown")
)

added = client.beta.sessions.resources.add(
    session_id=session.id,
    type="file",
    file_id=hints_file.id,
    mount_path="DEPLOY_HISTORY.md",
)
print(f"added resource {added.id} to session {session.id}")

attached = client.beta.sessions.resources.list(session_id=session.id)
print(f"{len(attached.data)} resources attached now")

client.beta.sessions.events.send(
    session_id=session.id,
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "There's a DEPLOY_HISTORY.md in your workspace now. "
                        "Read it and tell me whether it changes anything in "
                        "your earlier answer."
                    ),
                }
            ],
        }
    ],
)
print("\n--- follow-up with deploy history ---")
stream_until_end_turn(client, session.id)

# Detach the file now that the agent is done with it. `delete` here
# is the resource-detach verb, not the cookbook-wide archive.
client.beta.sessions.resources.delete(session_id=session.id, resource_id=added.id)
print("detached follow-up resource")

## 정리

In [ ]:
wait_for_idle_status(client, session.id)
client.beta.sessions.archive(session.id)
client.beta.environments.archive(env.id)
client.beta.agents.archive(agent.id)
print("archived")